# 1. Set up

In [2]:
import ee
import geemap

### Authenticate and initialize GEE account

In [3]:
ee.Authenticate()
ee.Initialize(project='musa-650')

### Add Landsat8 data to the map

I selected the coordinates of downtown Philadelphia as the POI and filtered the Landsat8 image covering this point. In addition, to avoid the influence of weather condition, I chose September because the weather is more likely to be clear, unlike the summer with heavy rains or the winter with snow. Then I sorted by cloud cover, selecting the smallest one. Finally, the selected image was loaded onto the map in true color band combination.

In [4]:
# define POI (downtown philly)
point = ee.Geometry.Point([-75.1652, 39.9526])

# choose Landsat 8 Collection 2 Level-2 image
image = (
    ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
    .filterBounds(point)
    .filterDate("2023-09-01", "2023-09-30")
    .sort("CLOUD_COVER")
    .first()
    .select(["SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"])  # 选择地表反射率波段
)

# set visualization parameters
vis_params = {
    "min": 5000,
    "max": 15000,
    "bands": ["SR_B4", "SR_B3", "SR_B2"]  # 近红外 (NIR), 红 (Red), 绿 (Green) 组成假色图像
}

# display map
Map = geemap.Map()
Map.centerObject(point, 10)
Map.addLayer(image, vis_params, "Landsat-8 (SR)")
Map


Map(center=[39.9526, -75.1652], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchD…

### Check image properties

In [129]:
ee.Date(image.get("system:time_start")).format("YYYY-MM-dd").getInfo()

'2023-09-01'

In [ ]:
image.get("CLOUD_COVER").getInfo()

1.11

#### Clip and download the Landsat image to google drive

The original image is too large, so we need to crop it to the appropriate size. I created a square ROI with a side length of 16km centered on the coordinates of downtown Philadelphia, then cropped the original image with it and exported it to Google Drive.

In [ ]:
point = ee.Geometry.Point([-75.1652, 39.9526])

# Create a buffer with a radius of 8km centered at the POI,
# and then draw the circumscribed square of the circle as the bound
buffer = point.buffer(8000)
region = buffer.bounds()
task = ee.batch.Export.image.toDrive(
    image=image,
    description="Landsat8_Image_philly",
    scale=30,
    region=region.getInfo()["coordinates"],
    maxPixels=1e13,
    fileFormat="GeoTIFF"
)

task.start()
print("Export started, check Google Drive.")


Export started, check Google Drive.


#### Display the clipped image on geemap

In [11]:
buffer = point.buffer(8000)
region = buffer.bounds()

clipped_image = image.clip(region)


vis_params = {
    "min": 5000,
    "max": 15000,
    "bands": ["SR_B4", "SR_B3", "SR_B2"]  # 近红外 (NIR), 红 (Red), 绿 (Green) 组成假色图像
}

Map = geemap.Map()
Map.centerObject(point, 12)
Map.addLayer(clipped_image, vis_params, "clipped_Landsat-8 (SR)")
Map


Map(center=[39.9526, -75.1652], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchD…

# 2. Data Collection and Feature Engineering

#### create land use label samples
I created the landuse labels manually on ArcGIS pro.
The label samples include 4 types: urban, bare, vegetation and water, and each type contains at least 100 samples. All the samples are *point*. The detailed infomation of my label samples are as follows:

---

**label - class - size**

0 - Urban - 100points

1 - Bare - 100points

2 - Vegetation -121points

3 - Water - 100points



#### Read the label sample data
I uploaded the sample data on GEE manually, and use ee.FeatureCollection to read it

In [6]:
asset_id = "projects/musa-650/assets/label_samples"

# read the label sample data from GEE
label_samples = ee.FeatureCollection(asset_id)

print(label_samples.getInfo())


{'type': 'FeatureCollection', 'columns': {'class': 'String', 'created_da': 'Long', 'created_us': 'String', 'label': 'Integer', 'system:index': 'String'}, 'version': 1741149209531619, 'id': 'projects/musa-650/assets/label_samples', 'properties': {'system:asset_size': 20067}, 'features': [{'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [-75.19954879891733, 39.916080543966046]}, 'id': '00000000000000000141', 'properties': {'class': 'bare', 'created_da': 1741161600000, 'created_us': '费靖淼', 'label': 1}}, {'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [-75.20093686827893, 39.91407004364258]}, 'id': '00000000000000000142', 'properties': {'class': 'bare', 'created_da': 1741161600000, 'created_us': '费靖淼', 'label': 1}}, {'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [-75.19456968284287, 39.913472656566114]}, 'id': '00000000000000000143', 'properties': {'class': 'bare', 'created_da': 1741161600000, 'created_us': '费靖淼', 'label': 1}}, {'type': 'F

#### Add NDVI, NDWI, MNDWI and DEM to the clipped Landsat image as new Bands

In [7]:
# NDVI
ndvi = clipped_image.normalizedDifference(["SR_B5", "SR_B4"]).rename("NDVI")

# NDBI
ndbi = clipped_image.normalizedDifference(["SR_B6", "SR_B5"]).rename("NDBI")

# MNDWI
mndwi = clipped_image.normalizedDifference(["SR_B3", "SR_B6"]).rename("MNDWI")

# get DEM and calculate slope
dem = ee.Image("USGS/SRTMGL1_003")  # use SRTM 30m DEM
slope = ee.Terrain.slope(dem)

# merge these features to Landsat image
clipped_image = clipped_image.addBands([ndvi, ndbi, mndwi, dem, slope])

# check the new bands
print(clipped_image.bandNames().getInfo())


['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'NDVI', 'NDBI', 'MNDWI', 'elevation', 'slope']


#### Apply sobel kernel on Band2, 3, 4, 5
Sobel kernel is a **gradient-based operator** that enhances edges, aiding in the identification of urban and vegetation boundaries. I applied it to Bands 2, 3, 4, and 5 because **these bands capture important spectral information related to visible and near-infrared reflectance, which helps in distinguishing edges more effectively**.

In [8]:
# create sobel kernel
sobel_kernel = ee.Kernel.sobel()

# apply sobel kernel on B2,3,4,5
sobel_b2 = clipped_image.select("SR_B2").convolve(sobel_kernel).rename("sobel_b2")
sobel_b3 = clipped_image.select("SR_B3").convolve(sobel_kernel).rename("sobel_b3")
sobel_b4 = clipped_image.select("SR_B4").convolve(sobel_kernel).rename("sobel_b4")
sobel_b5 = clipped_image.select("SR_B5").convolve(sobel_kernel).rename("sobel_b5")

# merge the sobel bands into clipped landsat image
clipped_image = clipped_image.addBands([sobel_b2, sobel_b3, sobel_b4, sobel_b5])
print(clipped_image.bandNames().getInfo())

['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'NDVI', 'NDBI', 'MNDWI', 'elevation', 'slope', 'sobel_b2', 'sobel_b3', 'sobel_b4', 'sobel_b5']


#### Normalized all bands to [0,1]

In [9]:
# calculate the min and max of each band
band_names = ["SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7",
              "NDVI", "NDBI", "MNDWI", "elevation", "slope", "sobel_b2", "sobel_b3", "sobel_b4", "sobel_b5", ]

def normalize_band(image, band):
    min_val = image.select(band).reduceRegion(
        reducer=ee.Reducer.min(),
        geometry=image.geometry(),
        scale=30,
        bestEffort=True
    ).getNumber(band)

    max_val = image.select(band).reduceRegion(
        reducer=ee.Reducer.max(),
        geometry=image.geometry(),
        scale=30,
        bestEffort=True
    ).getNumber(band)

    return image.expression(
        "(b - min) / (max - min)",
        {"b": image.select(band), "min": min_val, "max": max_val}
    ).rename(band + "_norm")

# normalize all bands
normalized_bands = [normalize_band(clipped_image, band) for band in band_names]

# combine the normalized bands into "normalized_clipped_image"
normalized_clipped_image = ee.Image.cat(normalized_bands)

print(normalized_clipped_image.bandNames().getInfo())


['SR_B1_norm', 'SR_B2_norm', 'SR_B3_norm', 'SR_B4_norm', 'SR_B5_norm', 'SR_B6_norm', 'SR_B7_norm', 'NDVI_norm', 'NDBI_norm', 'MNDWI_norm', 'elevation_norm', 'slope_norm', 'sobel_b2_norm', 'sobel_b3_norm', 'sobel_b4_norm', 'sobel_b5_norm']


In [118]:
# display the normalized_clipped_image and label_samples

vis_params = {
    "min": 0,
    "max": 1,
    "bands": ["SR_B4_norm", "SR_B3_norm", "SR_B2_norm"]  # 近红外 (NIR), 红 (Red), 绿 (Green) 组成假色图像
}

Map = geemap.Map()
Map.centerObject(point, 12)
Map.addLayer(normalized_clipped_image, vis_params, "Normalized Clipped Image")
Map.addLayer(label_samples, {}, "Label Samples")
Map




Map(center=[39.9526, -75.1652], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchD…

# 3. Model Training and Evaluation

#### Extract the features of the Landsat image pixel corresponding to the sample point

In [13]:
samples = normalized_clipped_image.sampleRegions(
    collection=label_samples,
    properties=["label", "class"],
    scale=30
)

print(samples.getInfo())

{'type': 'FeatureCollection', 'columns': {}, 'properties': {'band_order': ['SR_B1_norm', 'SR_B2_norm', 'SR_B3_norm', 'SR_B4_norm', 'SR_B5_norm', 'SR_B6_norm', 'SR_B7_norm', 'NDVI_norm', 'NDBI_norm', 'MNDWI_norm', 'elevation_norm', 'slope_norm', 'sobel_b2_norm', 'sobel_b3_norm', 'sobel_b4_norm', 'sobel_b5_norm']}, 'features': [{'type': 'Feature', 'geometry': None, 'id': '00000000000000000141_0', 'properties': {'MNDWI_norm': 0.6376508564028252, 'NDBI_norm': 0.5280226580300769, 'NDVI_norm': 0.20079942404700393, 'SR_B1_norm': 0.3999154269695282, 'SR_B2_norm': 0.3888495862483978, 'SR_B3_norm': 0.3840000033378601, 'SR_B4_norm': 0.4140256643295288, 'SR_B5_norm': 0.34542718529701233, 'SR_B6_norm': 0.3065149784088135, 'SR_B7_norm': 0.1728387326002121, 'class': 'bare', 'elevation_norm': 0.353658527135849, 'label': 1, 'slope_norm': 0.019934665170082008, 'sobel_b2_norm': 0.571195241937583, 'sobel_b3_norm': 0.5851641776756653, 'sobel_b4_norm': 0.5845991087041128, 'sobel_b5_norm': 0.593435047603872}

#### Split training and testing dataset, ensuring that the training set contains all types of labels.

In [71]:
# Ensure all labels are represented in the training data
all_labels = samples.aggregate_array("label").distinct()

def check_labels(feature):
    return all_labels.contains(feature.get("label"))

# Filter training data to ensure all labels are present
training_data = samples.randomColumn("random", seed = 123)
training_samples = training_data.filter(ee.Filter.lt("random", 0.7)).filter(ee.Filter.inList("label", all_labels))
testing_samples = training_data.filter(ee.Filter.gte("random", 0.7))

# Print the number of samples in each set and check if all labels are present in training set
print("Number of training samples:", training_samples.size().getInfo())
print("Number of testing samples:", testing_samples.size().getInfo())
print("Labels in training set:", training_samples.aggregate_array("label").distinct().getInfo())


Number of training samples: 304
Number of testing samples: 116
Labels in training set: [1, 0, 2, 3]


#### Model training
I trained 2 classifiers:

Classifier 1: using all features except for sobel bands

Classifier 2: using all features

In [47]:
classifier = ee.Classifier.smileRandomForest(numberOfTrees=10)

In [72]:
# train classifier 1 - without kernel filter feature
trained_classifier_1 = classifier.train(
    features=training_samples,
    classProperty="label",
    inputProperties=[
        "SR_B1_norm", "SR_B2_norm", "SR_B3_norm", "SR_B4_norm", "SR_B5_norm",
        "SR_B6_norm", "SR_B7_norm", "NDVI_norm", "NDBI_norm", "MNDWI_norm",
        "elevation_norm", "slope_norm"
    ]
)

In [73]:
# train classifier 2 - with sobel filter feature
trained_classifier_2 = classifier.train(
    features=training_samples,
    classProperty="label",
    inputProperties=[
        "SR_B1_norm", "SR_B2_norm", "SR_B3_norm", "SR_B4_norm", "SR_B5_norm",
        "SR_B6_norm", "SR_B7_norm", "NDBI_norm", "MNDWI_norm", "NDVI_norm",
        "elevation_norm", "slope_norm", "sobel_b2_norm", "sobel_b3_norm", "sobel_b4_norm", "sobel_b5_norm"
    ]
)

#### Accuracy assessment
Apply the 2 classifiers to the testing samples, and compare the accuracy.


In [131]:
# Apply the trained classifier to testing samples
classified_testing_samples = testing_samples.classify(trained_classifier_1)

# Compute the confusion matrix
confusion_matrix = classified_testing_samples.errorMatrix("label", "classification")


# Print the confusion matrix
print("Classifier 1 Accuracy:")
print("Confusion Matrix:")
print(confusion_matrix.getInfo())
# Get label order
label_order = confusion_matrix.order().getInfo()
print("Label Order in Confusion Matrix (Row/Column):", label_order)

# Compute the overall accuracy
overall_accuracy = confusion_matrix.accuracy().getInfo()
print(f"Overall Accuracy: {overall_accuracy:.4f}")
print("Precision:", confusion_matrix.consumersAccuracy().getInfo())
print("Recall:", confusion_matrix.producersAccuracy().getInfo())


Classifier 1 Accuracy:
Confusion Matrix:
[[23, 0, 0, 0], [3, 26, 0, 0], [0, 0, 28, 0], [0, 0, 0, 36]]
Label Order in Confusion Matrix (Row/Column): [0, 1, 2, 3]
Overall Accuracy: 0.9741
Precision: [[0.8846153846153846, 1, 1, 1]]
Recall: [[1], [0.896551724137931], [1], [1]]


In [132]:
# Apply the trained classifier to testing samples
classified_testing_samples = testing_samples.classify(trained_classifier_2)

# Compute the confusion matrix
confusion_matrix2 = classified_testing_samples.errorMatrix("label", "classification")

# Print the confusion matrix
print("Classifier 2 Accuracy:")
print("Confusion Matrix:")
print(confusion_matrix2.getInfo())
# Get label order
label_order = confusion_matrix2.order().getInfo()
print("Label Order in Confusion Matrix (Row/Column):", label_order)

# 4. Compute the overall accuracy
overall_accuracy = confusion_matrix2.accuracy().getInfo()
print(f"Overall Accuracy: {overall_accuracy:.4f}")
print("Precision:", confusion_matrix2.consumersAccuracy().getInfo())
print("Recall", confusion_matrix2.producersAccuracy().getInfo())


Classifier 2 Accuracy:
Confusion Matrix:
[[23, 0, 0, 0], [4, 25, 0, 0], [0, 0, 28, 0], [0, 0, 0, 36]]
Label Order in Confusion Matrix (Row/Column): [0, 1, 2, 3]
Overall Accuracy: 0.9655
Precision: [[0.8518518518518519, 1, 1, 1]]
Recall [[1], [0.8620689655172413], [1], [1]]


From the above accuracy evaluation, it can be seen that adding the Sobel filter as an input feature did not improve the model's performance. On the contrary, the accuracy slightly decreased.

#### Analyze importance of each feature

In [76]:
importance = trained_classifier_2.explain().get("importance")
print("Feature Importance: ", importance.getInfo())


Feature Importance:  {'MNDWI_norm': 1.4062416885196316, 'NDBI_norm': 1.1867682274370588, 'NDVI_norm': 1.2950629781118974, 'SR_B1_norm': 1.2955915052710836, 'SR_B2_norm': 0.505999496555704, 'SR_B3_norm': 1.9753118381375931, 'SR_B4_norm': 2.5541870985965276, 'SR_B5_norm': 2.0413920308792077, 'SR_B6_norm': 2.0437368118588495, 'SR_B7_norm': 2.2353206468589093, 'elevation_norm': 0.9429860979531486, 'slope_norm': 0.31999999999999995, 'sobel_b2_norm': 1.2105555555555556, 'sobel_b3_norm': 0.45486111111111116, 'sobel_b4_norm': 0.8911979270561872, 'sobel_b5_norm': 0.8028908212560387}


In [77]:
# Get feature importance as a dictionary
importance_dict = importance.getInfo()

# Sort features by importance in descending order
sorted_features = sorted(importance_dict.items(), key=lambda item: item[1], reverse=True)

# Print the sorted features
print("Sorted Features by Importance:")
for feature, importance_value in sorted_features:
    print(f"{feature}: {importance_value}")


Sorted Features by Importance:
SR_B4_norm: 2.5541870985965276
SR_B7_norm: 2.2353206468589093
SR_B6_norm: 2.0437368118588495
SR_B5_norm: 2.0413920308792077
SR_B3_norm: 1.9753118381375931
MNDWI_norm: 1.4062416885196316
SR_B1_norm: 1.2955915052710836
NDVI_norm: 1.2950629781118974
sobel_b2_norm: 1.2105555555555556
NDBI_norm: 1.1867682274370588
elevation_norm: 0.9429860979531486
sobel_b4_norm: 0.8911979270561872
sobel_b5_norm: 0.8028908212560387
SR_B2_norm: 0.505999496555704
sobel_b3_norm: 0.45486111111111116
slope_norm: 0.31999999999999995


Calculate the importance of each feature for **classifier2** and sort them in descending order. The most important feature is **B4**, while the least important is **slope**.  

I noticed that when training the model with different random seeds, the feature importance ranking varies each time, sometimes significantly.  

However, in general, after repeating the training multiple times, one common pattern emerges: **NDBI and B4, B5, B6, B7 are relatively important, while B2 and Sobel bands are relatively unimportant.**

#### Apply the classifier 1 to the clipped landsat image (ROI)

In [127]:
# Classify the image
classified_image = normalized_clipped_image.classify(trained_classifier_1)

# Display the classification result
Map = geemap.Map()
Map.centerObject(point, 12)
Map.addLayer(classified_image, {"min": 0, "max": 3, "palette": ['red', 'yellow', 'green', 'blue']}, "Classification")
Map


Map(center=[39.9526, -75.1652], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchD…

#### Export the classification image to geotiff

In [128]:
task = ee.batch.Export.image.toDrive(
    image=classified_image,
    description="Classified_Image_GeoTIFF",
    scale=30,
    region=region.getInfo()["coordinates"],
    maxPixels=1e13,
    fileFormat="GeoTIFF"
)

task.start()
print("Export of classified image to Google Drive as GeoTIFF started.")


Export of classified image to Google Drive as GeoTIFF started.


#### Visually compare the landcover data for my ROI with the corresponding landcover data from the European Space Agency.

Since my land cover classification differs from ESA's, a direct comparison is not straightforward. For example, ESA's classification includes **tree cover, shrubland, grassland, and cropland**, whereas in my land cover, all these categories are grouped under **"vegetation."** Additionally, ESA includes classes such as **snow and wetlands**, which are not present in my classification.  

To facilitate a better comparison, I merged the four vegetation-related classes in ESA's land cover, and assigned the veg, urban, bare, water as the same color as in my classification. Categories that are not included in my land cover were set to **white** for clarity.


---



**legend:**

urban - red

bare - yellow

vegetation - green

water - blue

others - white

In [102]:
esa_worldcover = (
    ee.ImageCollection("ESA/WorldCover/v200")
    .filterBounds(region)  # Use the previously defined region
    .first()

)

# Clip to the region of interest (you can use a buffer or directly select the ROI)
clipped_worldcover = esa_worldcover.clip(region)
clipped_worldcover.getInfo()

# Add ESA WorldCover data to the map
Map = geemap.Map()
Map.centerObject(point, 12)

# Define a color dictionary for each land cover label
color_map = {
    10: 'green',  # Tree Cover - Dark Green
    20: 'green',  # Shrubland - Light Brown
    30: 'green',  # Grassland - Yellow
    40: 'green',  # Cropland - Purple
    50: 'red',  # Built-up - Red
    60: 'yellow',  # Bare / Sparse vegetation - Grey
    70: 'white',  # Snow / Ice - White
    80: 'blue',  # Permanent Water Bodies - Blue
    90: 'white',  # Herbaceous Wetland - Cyan
    # 95: 'white',  # Mangroves - Greenish Teal
    # 100: 'white', # Moss / Lichen - Light Yellow
}

# Convert dictionary to a color palette (sorted by label values)
palette = [color_map[key] for key in sorted(color_map.keys())]

# Define visualization parameters
vis_params = {
    'min': 10,  # Minimum label value
    'max': 90,  # Maximum label value
    'palette': palette  # Apply custom colors
}

Map.addLayer(classified_image, {"min": 0, "max": 3, "palette": ['red', 'yellow', 'green', 'blue']}, "Classification")
Map.addLayer(clipped_worldcover, vis_params, "ESA WorldCover 2020")

# Display the map (if not already displayed)
Map


Map(center=[39.9526, -75.1652], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchD…

Comparing the two layers by check/uncheck the layers on the map, we can see that the classification results are very close overall. However, my classification **mistakenly classified some urban areas as bare areas**.

#### Explore the detailed vegetation classification of ESA.
Next, I separately extracted the vegetation areas, restored ESA’s original four vegetation types, and compared them with my classification. As can be seen, the overall extent of vegetation is quite similar, although my classification did not capture some fragmented vegetation patches in urban areas. The vegetation in the ESA image is indeed more precisely subdivided into different types, which can be regarded as a future improvement.

In [106]:
esa_worldcover = (
    ee.ImageCollection("ESA/WorldCover/v200")
    .filterBounds(region)  # Use the previously defined region
    .first()

)

# Clip to the region of interest (you can use a buffer or directly select the ROI)
clipped_worldcover = esa_worldcover.clip(region)
clipped_worldcover.getInfo()

# Add ESA WorldCover data to the map
Map = geemap.Map()
Map.centerObject(point, 12)

# Define a color dictionary for each land cover label
color_map = {
    10: '#006400',  # Tree Cover - Dark Green
    20: '#FFBB22',  # Shrubland - Light Brown
    30: '#FFFF4C',  # Grassland - Yellow
    40: '#F096FF',  # Cropland - Purple
    50: 'white',  # Built-up - Red
    60: 'white',  # Bare / Sparse vegetation - Grey
    70: 'white',  # Snow / Ice - White
    80: 'white',  # Permanent Water Bodies - Blue
    90: 'white',  # Herbaceous Wetland - Cyan
    # 95: 'white',  # Mangroves - Greenish Teal
    # 100: 'white', # Moss / Lichen - Light Yellow
}

# Convert dictionary to a color palette (sorted by label values)
palette = [color_map[key] for key in sorted(color_map.keys())]

# Define visualization parameters
vis_params = {
    'min': 10,  # Minimum label value
    'max': 90,  # Maximum label value
    'palette': palette  # Apply custom colors
}

Map.addLayer(classified_image, {"min": 0, "max": 3, "palette": ['white', 'white', 'green', 'white']}, "Classification")
Map.addLayer(clipped_worldcover, vis_params, "ESA WorldCover 2020")

# Display the map (if not already displayed)
Map


Map(center=[39.9526, -75.1652], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchD…

# 4. Reflection

#### a. What limitations did you run into when completing this assignment? What might you do differently if you repeated it, or what might you change if you had more time and/or resources?

Firstly, the sample size is too small. Secondly, the relatively low resolution of Landsat 8 (30m) means that many pixels represent a mixture of different land cover types, which not only makes sample selection more challenging but also reduces the classification accuracy. Lastly, the classification of the four land use categories is too broad, and it would be beneficial to introduce more detailed subcategories, similar to those in the ESA classification.


---



#### b. What was the impact of feature engineering? Which layers most contributed to the model? Did you expect this? Why or why not?

The feature engineering, particularly the normalization process, had a significant impact on the model. Normalization ensures that all input features are on the same scale, preventing any single feature from dominating the model's learning process. This improves model performance by allowing more effective learning and better interpretation of the relationships between features. And adding NDVI, NDBI MNDWI, elevation and slope also enhances the model by incorporating additional information that captures vegetation, built-up areas, water bodies, and topographic variations.

**Band 4 (Red)** contributed the most to the model's performance. This is quite reasonable because the red band is highly sensitive to vegetation and can help distinguish different land cover types. The red band is often used in vegetation indices, such as NDVI, and plays a crucial role in identifying differences in land cover types.

However, the contribution of slope and other sobel features was less than expected. I anticipated that topographic features would play a larger role, but the model was more sensitive to the spectral bands, which suggests that land cover classification in this case was primarily driven by spectral differences rather than topographic features.



---


#### c. Did you find it difficult to create the training data by hand? Did you notice any issues with class imbalance? If so, how might you resolve this in the future (hint: consider a different sampling technique).

Some classes of samples were easy to create by hand, such as water bodies, because they are very distinct. However, others were more challenging, especially **bare areas**, which are difficult to distinguish and often small and fragmented. This also led to issues with **class imbalance**, as some classes were underrepresented. In the future, I could resolve this using a different sampling technique, such as **stratified sampling (which ensures that each class in the dataset is represented proportionally in both the training and testing sets.)** or **oversampling (ensures that each class in the dataset is represented proportionally in both the training and testing sets.)** the underrepresented classes to ensure a more balanced dataset.


---


#### d. Did your model perform better on one class than another? Why? Can you think of a reason that this might be good or bad depending on the context?

The model performed better on **vegetation** and **water** than on **urban** and **bare** classes (the precision and recall of vegetation and water are all 1). This is likely because vegetation and water have more distinct and easily recognizable spectral signatures, making them easier for the model to differentiate. On the other hand, urban and bare areas can have more complex and mixed spectral characteristics, which may make classification more challenging.

In some contexts, this could be a good thing, as **vegetation** and **water** are often of greater interest in land cover classification, such as in environmental monitoring or conservation. However, it could be a problem if accurate classification of **urban** or **bare** areas is critical for the application, such as in urban planning or infrastructure development. In such cases, improving the model's performance on these underperforming classes would be necessary.

> The code debugging and text refinement in this project were assisted by ChatGPT, which provided valuable suggestions that greatly enhanced my work efficiency.
>
> Source: OpenAI ChatGPT, accessed on March 5, 2025.
